# Notebook 08: Noise-Aware Out-of-Distribution (OOD) Detector (V2 Scorer)

**Role 2: Trust Engine Engineer**

This notebook investigates and implements the **V2 Noise-Aware Mahalanobis Detector**.
- Integrates physical measurement noise standard deviations ($\sigma_\text{raw}$) from the approved `primary_1sigma` profile in `backend/models/noise_model.json` via `backend/noise.py`.
- Formulates the measurement-noise augmented covariance matrix:
  $$\Sigma_\text{v2} = \Sigma_\text{LW} + \text{diag}(\text{var}_z)$$
  where $\text{var}_z = (\sigma_\text{raw} / \text{scale})^2$ represents the sensor variance projected into standardized $z$-space.
- Maintains strict zero-leakage protocol: all statistics, noise-aware covariances, and OOF threshold distributions are fitted and evaluated strictly on training rows (`train_idx.npy`). No validation or test rows are used for threshold selection or model fitting.

### Sections
1. **Setup and V0 Reproduction**: Verifies environment, loads data splits, and reproduces baseline (V0) Ledoit-Wolf Mahalanobis detector metrics (E5 synthetic unknowns, LOCO cross-validation, and OOF percentiles) matching locked project benchmarks.
2. **V2 Noise-Aware Scorer**: Implements `make_v2_stats` and `compute_v2_scores`, augmenting per-crop covariance matrices with physical measurement variances.
3. **V2 OOF Thresholds**: Computes 5-fold stratified out-of-fold reference distance distributions on `X_train` with the V2 scorer.
4. **Physical Noise Magnitudes Inspection**: Displays 22-crop $\times$ 7-feature tables of physical noise magnitudes $\sigma_\text{raw}$ and standardized variances $\text{var}_z$.


## Section 1: Setup and V0 Reproduction

We import required numerical and ML libraries, load canonical feature specifications, and verify reproducibility with `SEED = 42`.
We reproduce baseline V0 per-crop Ledoit-Wolf Mahalanobis detector benchmarks to guarantee exact parity with `notebooks/04_ood_eval.ipynb`.


In [1]:
import os
import sys
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

# Ensure reproducible random state
SEED = 42
np.random.seed(SEED)

# Canonical 7-feature order frozen across API contract and engine
CANONICAL_FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
print(f"Canonical feature order: {CANONICAL_FEATURES}")

# Add repository root to Python path for backend imports
repo_root = Path("..").resolve() if Path("..", "backend").exists() else Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend.noise import load_noise_config, compute_feature_sigmas

# Load approved noise profile configuration
noise_config_path = repo_root / "backend" / "models" / "noise_model.json"
noise_config = load_noise_config(noise_config_path)
PROFILE_CFG = noise_config["profiles"]["primary_1sigma"]
print(f"Loaded approved noise profile: 'primary_1sigma' ({PROFILE_CFG['description'][:65]}...)")


Canonical feature order: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
Loaded approved noise profile: 'primary_1sigma' (Primary approved noise model for stability analysis (E8), sensiti...)


In [2]:
# 1. Load dataset
data_path = os.path.join("..", "data", "Crop_recommendation.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "Crop_recommendation.csv")

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# 2. Load split indices (strictly train and val; test_idx is never loaded or used)
split_dir = os.path.join("..", "data", "splits")
if not os.path.exists(split_dir):
    split_dir = os.path.join("data", "splits")

train_idx = np.load(os.path.join(split_dir, "train_idx.npy"))
val_idx = np.load(os.path.join(split_dir, "val_idx.npy"))

print(f"Split sizes: Train={len(train_idx)}, Val={len(val_idx)}")

# 3. Verify disjointness and split sizes
assert set(train_idx).isdisjoint(set(val_idx)), "ERROR: Train and Val overlap!"
assert len(train_idx) == 1540, f"Expected 1540 train rows, got {len(train_idx)}"
assert len(val_idx) == 330, f"Expected 330 val rows, got {len(val_idx)}"

# 4. Extract feature matrices and labels
X = df[CANONICAL_FEATURES].values
y = df['label'].values

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

classes = sorted(list(np.unique(y_train)))
print(f"Verified {len(classes)} classes with exactly {len(X_train) // len(classes)} train rows per class.")
print(f"Shapes: X_train={X_train.shape}, X_val={X_val.shape}")
print("Test split status: COMPLETELY EXCLUDED (Zero leakage enforced).")


Loaded dataset: 2200 rows, 8 columns
Split sizes: Train=1540, Val=330
Verified 22 classes with exactly 70 train rows per class.
Shapes: X_train=(1540, 7), X_val=(330, 7)
Test split status: COMPLETELY EXCLUDED (Zero leakage enforced).


In [3]:
# 1. Fit Global Feature Scaler on X_train ONLY
scaler_v0 = StandardScaler()
X_train_scaled = scaler_v0.fit_transform(X_train)
X_val_scaled = scaler_v0.transform(X_val)

# 2. Fit Per-Crop Ledoit-Wolf Covariance Estimators (V0 Baseline)
crop_stats_v0 = {}
shrinkages_v0 = {}

for crop in classes:
    mask = (y_train == crop)
    Xc_scaled = X_train_scaled[mask]
    
    lw = LedoitWolf(assume_centered=False)
    lw.fit(Xc_scaled)
    
    crop_stats_v0[crop] = {
        'mean': lw.location_,
        'cov': lw.covariance_,
        'inv_cov': lw.precision_,
        'shrinkage': float(lw.shrinkage_)
    }
    shrinkages_v0[crop] = float(lw.shrinkage_)

avg_shrinkage = np.mean(list(shrinkages_v0.values()))
print(f"V0 Per-Crop Ledoit-Wolf fitted. Average shrinkage: {avg_shrinkage:.4f} (range: {min(shrinkages_v0.values()):.4f} - {max(shrinkages_v0.values()):.4f})")

def compute_mahalanobis_scores(X_input_scaled, stats_dict, crop_classes):
    n_samples = len(X_input_scaled)
    all_d2 = np.zeros((n_samples, len(crop_classes)))
    for idx, crop in enumerate(crop_classes):
        diff = X_input_scaled - stats_dict[crop]['mean']
        prec = stats_dict[crop]['inv_cov']
        all_d2[:, idx] = np.sum((diff @ prec) * diff, axis=1)
    return np.min(all_d2, axis=1)


V0 Per-Crop Ledoit-Wolf fitted. Average shrinkage: 0.0758 (range: 0.0193 - 0.1807)


In [4]:
# 1. Stratified Validation Partitioning (10 calib, 5 eval per crop)
rng_val = np.random.default_rng(seed=SEED)
calib_idx_list = []
eval_idx_list = []

for crop in classes:
    crop_pos = np.where(y_val == crop)[0]
    shuffled = rng_val.permutation(crop_pos)
    calib_idx_list.extend(shuffled[:10])
    eval_idx_list.extend(shuffled[10:])

calib_sub_idx = np.array(calib_idx_list)
eval_sub_idx = np.array(eval_idx_list)

X_val_calib, y_val_calib = X_val[calib_sub_idx], y_val[calib_sub_idx]
X_val_eval, y_val_eval = X_val[eval_sub_idx], y_val[eval_sub_idx]

X_val_calib_scaled = X_val_scaled[calib_sub_idx]
X_val_eval_scaled = X_val_scaled[eval_sub_idx]

print(f"Validation Partitioning: Calibration Set = {len(X_val_calib)} rows, Evaluation Set = {len(X_val_eval)} rows")
print(f"Shapes: X_val_calib={X_val_calib.shape}, X_val_eval={X_val_eval.shape}")

# 2. 5-Fold Stratified Out-of-Fold (OOF) Reference Scores on X_train (1,540 rows)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_v0_scores = np.zeros(len(X_train))

for tr_fold_idx, oof_fold_idx in skf.split(X_train, y_train):
    X_tr_f, y_tr_f = X_train[tr_fold_idx], y_train[tr_fold_idx]
    X_oof_f = X_train[oof_fold_idx]
    
    sc_f = StandardScaler().fit(X_tr_f)
    X_tr_f_sc = sc_f.transform(X_tr_f)
    X_oof_f_sc = sc_f.transform(X_oof_f)
    
    stats_f = {}
    for crop in classes:
        mask = (y_tr_f == crop)
        lw_f = LedoitWolf(assume_centered=False).fit(X_tr_f_sc[mask])
        stats_f[crop] = {'mean': lw_f.location_, 'inv_cov': lw_f.precision_}
        
    for i, x_sc in enumerate(X_oof_f_sc):
        min_d = np.inf
        for crop in classes:
            diff = x_sc - stats_f[crop]['mean']
            d2 = float(diff @ stats_f[crop]['inv_cov'] @ diff)
            if d2 < min_d:
                min_d = d2
        oof_v0_scores[oof_fold_idx[i]] = min_d

print("V0 OOF Reference Scores (1,540 samples):")
print(f"  Median: {np.median(oof_v0_scores):.4f}")
print(f"  p90:    {np.percentile(oof_v0_scores, 90.0):.4f}")
print(f"  p95:    {np.percentile(oof_v0_scores, 95.0):.4f}")
print(f"  p97:    {np.percentile(oof_v0_scores, 97.0):.4f}")
print(f"  p99:    {np.percentile(oof_v0_scores, 99.0):.4f}")
print(f"  p99.5:  {np.percentile(oof_v0_scores, 99.5):.4f}")


Validation Partitioning: Calibration Set = 220 rows, Evaluation Set = 110 rows
Shapes: X_val_calib=(220, 7), X_val_eval=(110, 7)


V0 OOF Reference Scores (1,540 samples):
  Median: 5.7980
  p90:    9.5342
  p95:    10.7842
  p97:    11.7121
  p99:    12.9220
  p99.5:  13.7512


In [5]:
# 1. Unknown Generators
def generate_box_uniform(n, bounds_min, bounds_max, seed):
    rng = np.random.default_rng(seed)
    return rng.uniform(bounds_min, bounds_max, size=(n, len(bounds_min)))

def generate_shuffled_columns(X_source, n, seed):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X_source), size=n, replace=True)
    sampled = X_source[idx].copy()
    for col in range(sampled.shape[1]):
        rng.shuffle(sampled[:, col])
    return sampled

def evaluate_detector(scores_known, scores_unknown):
    y_true = np.array([0] * len(scores_known) + [1] * len(scores_unknown))
    scores_combined = np.concatenate([scores_known, scores_unknown])
    
    auroc = roc_auc_score(y_true, scores_combined)
    fpr, tpr, _ = roc_curve(y_true, scores_combined)
    
    idx_95 = np.where(tpr >= 0.95)[0]
    fpr_at_95 = fpr[idx_95[0]] if len(idx_95) > 0 else 1.0
    return auroc, fpr_at_95

train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)

box_eval = generate_box_uniform(1000, train_min, train_max, seed=202)
shuf_eval = generate_shuffled_columns(X_val_eval, 1000, seed=404)

print(f"Generated unknowns: box_eval shape={box_eval.shape}, shuf_eval shape={shuf_eval.shape}")

box_eval_scaled = scaler_v0.transform(box_eval)
shuf_eval_scaled = scaler_v0.transform(shuf_eval)

m_known_v0 = compute_mahalanobis_scores(X_val_eval_scaled, crop_stats_v0, classes)
m_box_v0 = compute_mahalanobis_scores(box_eval_scaled, crop_stats_v0, classes)
m_shuf_v0 = compute_mahalanobis_scores(shuf_eval_scaled, crop_stats_v0, classes)

auc_box_v0, fpr_box_v0 = evaluate_detector(m_known_v0, m_box_v0)
auc_shuf_v0, fpr_shuf_v0 = evaluate_detector(m_known_v0, m_shuf_v0)

print(f"V0 Box Unknowns:      AUROC = {auc_box_v0:.4f}, FPR@95% = {fpr_box_v0:.4f}")
print(f"V0 Shuffled Unknowns: AUROC = {auc_shuf_v0:.4f}, FPR@95% = {fpr_shuf_v0:.4f}")


Generated unknowns: box_eval shape=(1000, 7), shuf_eval shape=(1000, 7)
V0 Box Unknowns:      AUROC = 0.9996, FPR@95% = 0.0000
V0 Shuffled Unknowns: AUROC = 0.9778, FPR@95% = 0.0727


In [6]:
loco_records_v0 = []

for held_out in classes:
    # 1. Fit scaler and detectors exclusively on training rows of remaining 21 crops (1,470 rows)
    train_mask = (y_train != held_out)
    X_tr_loco = X_train[train_mask]
    y_tr_loco = y_train[train_mask]
    
    sc_loco = StandardScaler().fit(X_tr_loco)
    X_tr_loco_sc = sc_loco.transform(X_tr_loco)
    
    crops_loco = [c for c in classes if c != held_out]
    stats_loco = {}
    for c in crops_loco:
        Xc_sc = X_tr_loco_sc[y_tr_loco == c]
        lw_loco = LedoitWolf(assume_centered=False).fit(Xc_sc)
        stats_loco[c] = {'mean': lw_loco.location_, 'inv_cov': lw_loco.precision_}
        
    def score_loco_batch(X_sc):
        min_d = np.full(len(X_sc), np.inf)
        for c in crops_loco:
            diff = X_sc - stats_loco[c]['mean']
            d2 = np.sum((diff @ stats_loco[c]['inv_cov']) * diff, axis=1)
            min_d = np.minimum(min_d, d2)
        return min_d

    # 2. Known: validation rows of 21 known crops (315 rows, y=0)
    val_known_mask = (y_val != held_out)
    X_val_known_sc = sc_loco.transform(X_val[val_known_mask])
    scores_known = score_loco_batch(X_val_known_sc)
    
    # 3. Unknown: validation rows strictly from held-out crop (15 rows, y=1)
    val_held_out_mask = (y_val == held_out)
    X_held_out_sc = sc_loco.transform(X_val[val_held_out_mask])
    scores_unknown = score_loco_batch(X_held_out_sc)
    
    auc_loco, fpr_loco = evaluate_detector(scores_known, scores_unknown)
    loco_records_v0.append({
        'crop': held_out,
        'auroc': float(auc_loco),
        'fpr_95': float(fpr_loco),
        'median_dist': float(np.median(scores_unknown))
    })

loco_df_v0 = pd.DataFrame(loco_records_v0)
mean_loco_auc_v0 = float(loco_df_v0['auroc'].mean())
std_loco_auc_v0 = float(loco_df_v0['auroc'].std())
mean_loco_fpr_v0 = float(loco_df_v0['fpr_95'].mean())
std_loco_fpr_v0 = float(loco_df_v0['fpr_95'].std())

print("V0 LOCO Results across 22 Crops:")
print(f"  Mean AUROC:     {mean_loco_auc_v0:.4f} +/- {std_loco_auc_v0:.4f}")
print(f"  Mean FPR@95%:   {mean_loco_fpr_v0:.4f} +/- {std_loco_fpr_v0:.4f}")


V0 LOCO Results across 22 Crops:
  Mean AUROC:     0.9797 +/- 0.0526
  Mean FPR@95%:   0.1046 +/- 0.2338


In [7]:
# Compute V0 OOF Percentiles
v0_p90 = float(np.percentile(oof_v0_scores, 90.0))
v0_p95 = float(np.percentile(oof_v0_scores, 95.0))
v0_p97 = float(np.percentile(oof_v0_scores, 97.0))
v0_p99 = float(np.percentile(oof_v0_scores, 99.0))
v0_p99_5 = float(np.percentile(oof_v0_scores, 99.5))

v0_comparison_rows = [
    {"Metric": "Box AUROC", "Computed": f"{auc_box_v0:.4f}", "Expected": "0.9996", "Status": "PASS"},
    {"Metric": "Box FPR@95%", "Computed": f"{fpr_box_v0:.4f}", "Expected": "0.0000", "Status": "PASS"},
    {"Metric": "Shuffled AUROC", "Computed": f"{auc_shuf_v0:.4f}", "Expected": "0.9778", "Status": "PASS"},
    {"Metric": "Shuffled FPR@95%", "Computed": f"{fpr_shuf_v0:.4f}", "Expected": "0.0727", "Status": "PASS"},
    {"Metric": "LOCO Mean AUROC", "Computed": f"{mean_loco_auc_v0:.4f}", "Expected": "0.9797", "Status": "PASS"},
    {"Metric": "LOCO Std AUROC", "Computed": f"{std_loco_auc_v0:.4f}", "Expected": "0.0526", "Status": "PASS"},
    {"Metric": "LOCO Mean FPR@95%", "Computed": f"{mean_loco_fpr_v0:.4f}", "Expected": "0.1046", "Status": "PASS"},
    {"Metric": "LOCO Std FPR@95%", "Computed": f"{std_loco_fpr_v0:.4f}", "Expected": "0.2338", "Status": "PASS"},
    {"Metric": "OOF Percentile p90", "Computed": f"{v0_p90:.2f}", "Expected": "9.53", "Status": "PASS"},
    {"Metric": "OOF Percentile p95", "Computed": f"{v0_p95:.2f}", "Expected": "10.78", "Status": "PASS"},
    {"Metric": "OOF Percentile p97", "Computed": f"{v0_p97:.2f}", "Expected": "11.71", "Status": "PASS"},
    {"Metric": "OOF Percentile p99", "Computed": f"{v0_p99:.2f}", "Expected": "12.92", "Status": "PASS"},
    {"Metric": "OOF Percentile p99.5", "Computed": f"{v0_p99_5:.2f}", "Expected": "13.75", "Status": "PASS"},
]

df_v0_comp = pd.DataFrame(v0_comparison_rows)
print("=== V0 REPRODUCTION COMPARISON TABLE ===")
print(df_v0_comp.to_string(index=False))

# Strict Parity Assertions (4 decimals for classification, 2 decimals for OOF percentiles)
np.testing.assert_almost_equal(auc_box_v0, 0.9996, decimal=4, err_msg="Box AUROC mismatch")
np.testing.assert_almost_equal(fpr_box_v0, 0.0000, decimal=4, err_msg="Box FPR@95% mismatch")
np.testing.assert_almost_equal(auc_shuf_v0, 0.9778, decimal=4, err_msg="Shuffled AUROC mismatch")
np.testing.assert_almost_equal(fpr_shuf_v0, 0.0727, decimal=4, err_msg="Shuffled FPR@95% mismatch")
np.testing.assert_almost_equal(mean_loco_auc_v0, 0.9797, decimal=4, err_msg="LOCO Mean AUROC mismatch")
np.testing.assert_almost_equal(std_loco_auc_v0, 0.0526, decimal=4, err_msg="LOCO Std AUROC mismatch")
np.testing.assert_almost_equal(mean_loco_fpr_v0, 0.1046, decimal=4, err_msg="LOCO Mean FPR@95% mismatch")
np.testing.assert_almost_equal(std_loco_fpr_v0, 0.2338, decimal=4, err_msg="LOCO Std FPR@95% mismatch")

np.testing.assert_almost_equal(v0_p97, 11.71, decimal=2, err_msg="p97 OOF percentile mismatch")
np.testing.assert_almost_equal(v0_p99, 12.92, decimal=2, err_msg="p99 OOF percentile mismatch")
np.testing.assert_almost_equal(v0_p99_5, 13.75, decimal=2, err_msg="p99.5 OOF percentile mismatch")

print()
print("ALL V0 REPRODUCTION ASSERTIONS PASSED TO SPECIFIED TOLERANCE.")


=== V0 REPRODUCTION COMPARISON TABLE ===
              Metric Computed Expected Status
           Box AUROC   0.9996   0.9996   PASS
         Box FPR@95%   0.0000   0.0000   PASS
      Shuffled AUROC   0.9778   0.9778   PASS
    Shuffled FPR@95%   0.0727   0.0727   PASS
     LOCO Mean AUROC   0.9797   0.9797   PASS
      LOCO Std AUROC   0.0526   0.0526   PASS
   LOCO Mean FPR@95%   0.1046   0.1046   PASS
    LOCO Std FPR@95%   0.2338   0.2338   PASS
  OOF Percentile p90     9.53     9.53   PASS
  OOF Percentile p95    10.78    10.78   PASS
  OOF Percentile p97    11.71    11.71   PASS
  OOF Percentile p99    12.92    12.92   PASS
OOF Percentile p99.5    13.75    13.75   PASS

ALL V0 REPRODUCTION ASSERTIONS PASSED TO SPECIFIED TOLERANCE.


## Section 2: V2 Scorer Implementation

We implement the `make_v2_stats` function and the corresponding scoring function.
Given training data:
1. Standardizes features with `StandardScaler`.
2. Estimates empirical covariance with `LedoitWolf(assume_centered=False)`.
3. Inverts crop mean $\mu_z$ back to raw units $\mu_\text{raw} = \text{scaler.inverse_transform}(\mu_z)$.
4. Evaluates physical measurement standard deviations $\sigma_\text{raw} = \text{compute_feature_sigmas}(\mu_\text{raw}, \text{profile_cfg})$ from the approved 1-sigma profile in `noise_model.json`.
5. Projects measurement noise variances into standardized space:
   $$\text{var}_z = \left(\frac{\sigma_\text{raw}}{\text{scale}}\right)^2$$
6. Augments covariance matrix:
   $$\Sigma_\text{v2} = \Sigma_\text{LW} + \text{diag}(\text{var}_z)$$
7. Verifies symmetry and strict positive definiteness via Cholesky decomposition ($L = \text{cholesky}(\Sigma_\text{v2})$), and computes precision matrix $\Sigma_\text{v2}^{-1}$.


In [8]:
def make_v2_stats(X_fit_raw, y_fit, classes_fit, profile_cfg=PROFILE_CFG):
    """Fit V2 noise-aware per-crop statistics on raw feature inputs."""
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit_raw)
    
    crop_stats = {}
    for crop in classes_fit:
        mask = (y_fit == crop)
        Xc_scaled = X_fit_scaled[mask]
        
        # Fit Ledoit-Wolf shrinkage estimator on standardized training data
        lw = LedoitWolf(assume_centered=False)
        lw.fit(Xc_scaled)
        mean_z = lw.location_
        cov_LW = lw.covariance_
        
        # Invert crop center back to raw physical units
        mean_raw = scaler.inverse_transform(mean_z.reshape(1, -1))[0]
        
        # Compute approved 1-sigma measurement noise at crop center
        sigma_raw = compute_feature_sigmas(mean_raw, profile_cfg)
        
        # Project physical noise variance to standardized z-space
        var_z = (sigma_raw / scaler.scale_) ** 2
        
        # Augment covariance with independent measurement noise variance
        Sigma_v2 = cov_LW + np.diag(var_z)
        
        # Verify symmetry and strict positive definiteness via Cholesky decomposition
        assert np.allclose(Sigma_v2, Sigma_v2.T, atol=1e-12), f"Sigma_v2 not symmetric for crop {crop}"
        np.linalg.cholesky(Sigma_v2)  # Raises LinAlgError if not positive definite
        
        # Compute precision matrix
        inv_v2 = np.linalg.inv(Sigma_v2)
        
        crop_stats[crop] = {
            'mean_z': mean_z,
            'mean_raw': mean_raw,
            'sigma_raw': sigma_raw,
            'var_z': var_z,
            'cov_LW': cov_LW,
            'Sigma_v2': Sigma_v2,
            'inv': inv_v2,
            'shrinkage': float(lw.shrinkage_)
        }
        
    return scaler, crop_stats

def compute_v2_scores(X_input_raw, scaler_v2, crop_stats_v2, classes_v2):
    """Compute minimum squared Mahalanobis distance under V2 noise-aware covariance."""
    X_scaled = scaler_v2.transform(X_input_raw)
    n_samples = len(X_scaled)
    all_d2 = np.zeros((n_samples, len(classes_v2)))
    for idx, crop in enumerate(classes_v2):
        diff = X_scaled - crop_stats_v2[crop]['mean_z']
        prec = crop_stats_v2[crop]['inv']
        all_d2[:, idx] = np.sum((diff @ prec) * diff, axis=1)
    return np.min(all_d2, axis=1)

# Fit V2 statistics on full training set (1,540 rows)
scaler_v2_full, crop_stats_v2_full = make_v2_stats(X_train, y_train, classes)
print(f"V2 noise-aware statistics fitted on full X_train ({len(X_train)} samples).")
print(f"Verified Cholesky positive definiteness across all {len(classes)} crop covariance matrices.")


V2 noise-aware statistics fitted on full X_train (1540 samples).
Verified Cholesky positive definiteness across all 22 crop covariance matrices.


## Section 3: V2 Out-of-Fold (OOF) Reference Thresholds

We repeat the 5-fold Stratified Cross-Validation out-of-fold reference scoring on `X_train` (1,540 rows, seed 42) using `make_v2_stats`.
- In each fold, `make_v2_stats` is fitted strictly on the 4 training folds (`X_tr_f`, 1,232 rows).
- Out-of-fold squared Mahalanobis distances are evaluated on `X_oof_f` (308 rows).
- Reference percentiles are derived entirely from training data. Zero validation rows are inspected or utilized.


In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_v2_scores = np.zeros(len(X_train))

for tr_fold_idx, oof_fold_idx in skf.split(X_train, y_train):
    X_tr_f, y_tr_f = X_train[tr_fold_idx], y_train[tr_fold_idx]
    X_oof_f = X_train[oof_fold_idx]
    
    sc_f, stats_f = make_v2_stats(X_tr_f, y_tr_f, classes, profile_cfg=PROFILE_CFG)
    d2_oof = compute_v2_scores(X_oof_f, sc_f, stats_f, classes)
    oof_v2_scores[oof_fold_idx] = d2_oof

v2_median = float(np.median(oof_v2_scores))
v2_p90 = float(np.percentile(oof_v2_scores, 90.0))
v2_p95 = float(np.percentile(oof_v2_scores, 95.0))
v2_p97 = float(np.percentile(oof_v2_scores, 97.0))
v2_p99 = float(np.percentile(oof_v2_scores, 99.0))
v2_p99_5 = float(np.percentile(oof_v2_scores, 99.5))

v0_median = float(np.median(oof_v0_scores))

oof_percentile_comparison = pd.DataFrame([
    {"Percentile": "Median (p50)", "V0 OOF d2": round(v0_median, 4), "V2 OOF d2": round(v2_median, 4)},
    {"Percentile": "90.0th (p90)", "V0 OOF d2": round(v0_p90, 4), "V2 OOF d2": round(v2_p90, 4)},
    {"Percentile": "95.0th (p95)", "V0 OOF d2": round(v0_p95, 4), "V2 OOF d2": round(v2_p95, 4)},
    {"Percentile": "97.0th (p97)", "V0 OOF d2": round(v0_p97, 4), "V2 OOF d2": round(v2_p97, 4)},
    {"Percentile": "99.0th (p99)", "V0 OOF d2": round(v0_p99, 4), "V2 OOF d2": round(v2_p99, 4)},
    {"Percentile": "99.5th (p99.5)", "V0 OOF d2": round(v0_p99_5, 4), "V2 OOF d2": round(v2_p99_5, 4)},
])

print("=== OOF REFERENCE PERCENTILE COMPARISON (1,540 TRAIN ROWS) ===")
print(oof_percentile_comparison.to_string(index=False))


=== OOF REFERENCE PERCENTILE COMPARISON (1,540 TRAIN ROWS) ===
    Percentile  V0 OOF d2  V2 OOF d2
  Median (p50)     5.7980     3.6491
  90.0th (p90)     9.5342     6.1925
  95.0th (p95)    10.7842     7.0610
  97.0th (p97)    11.7121     7.5904
  99.0th (p99)    12.9220     8.6355
99.5th (p99.5)    13.7512     9.1759


## Section 4: Physical Noise Magnitudes Inspection (22 Crops x 7 Features)

We inspect the physical measurement standard deviations $\sigma_\text{raw}$ and standardized measurement variances $\text{var}_z = (\sigma_\text{raw} / \text{scale})^2$ for the 22 target crops from the full-train V2 fit.


In [10]:
rows_sigma = []
rows_var_z = []

for crop in classes:
    stats = crop_stats_v2_full[crop]
    rows_sigma.append([crop] + list(np.round(stats['sigma_raw'], 4)))
    rows_var_z.append([crop] + list(np.round(stats['var_z'], 4)))

df_sigma = pd.DataFrame(rows_sigma, columns=['crop'] + CANONICAL_FEATURES)
df_var_z = pd.DataFrame(rows_var_z, columns=['crop'] + CANONICAL_FEATURES)

print("=== 1. PHYSICAL SENSOR NOISE 1-SIGMA: sigma_raw (Dataset Units) ===")
print(df_sigma.to_string(index=False))

print()
print("=== 2. STANDARDIZED NOISE VARIANCE: var_z = (sigma_raw / scale)^2 ===")
print(df_var_z.to_string(index=False))

# Feature-wise Summary across all 22 crops
summary_data = []
for feat in CANONICAL_FEATURES:
    sigmas_feat = df_sigma[feat].values
    var_z_feat = df_var_z[feat].values
    summary_data.append({
        "Feature": feat,
        "Min sigma_raw": round(float(np.min(sigmas_feat)), 4),
        "Mean sigma_raw": round(float(np.mean(sigmas_feat)), 4),
        "Max sigma_raw": round(float(np.max(sigmas_feat)), 4),
        "Mean var_z": round(float(np.mean(var_z_feat)), 4),
    })

df_summary = pd.DataFrame(summary_data)
print()
print("=== FEATURE NOISE SUMMARY ACROSS 22 CROPS ===")
print(df_summary.to_string(index=False))


=== 1. PHYSICAL SENSOR NOISE 1-SIGMA: sigma_raw (Dataset Units) ===
       crop       N       P       K  temperature  humidity  ph  rainfall
      apple  2.0814 13.3771 20.0100          1.5       5.0 0.3   11.2031
     banana 10.0786  8.1486  4.9786          1.5       5.0 0.3   10.4905
  blackgram  4.0129  6.7571  2.0000          1.5       5.0 0.3    6.8050
   chickpea  3.9657  6.8214  7.9686          1.5       5.0 0.3    7.9801
    coconut  2.2743  2.0000  3.0529          1.5       5.0 0.3   17.3903
     coffee 10.1600  2.8586  2.9686          1.5       5.0 0.3   15.7246
     cotton 11.7300  4.6529  2.0000          1.5       5.0 0.3    7.9997
     grapes  2.4214 13.1914 20.0329          1.5       5.0 0.3    6.9771
       jute  7.7814  4.7086  4.0157          1.5       5.0 0.3   17.6082
kidneybeans  2.0000  6.7414  2.0000          1.5       5.0 0.3   10.5396
     lentil  2.0000  6.8857  2.0000          1.5       5.0 0.3    5.0000
      maize  7.7329  4.8457  2.0043          1.5       5

## Section 5: Experiment E5 Synthetic Unknowns Benchmark (Before / After)

We evaluate the baseline V0 detector, the V2 noise-aware detector, and the 1D range check naive baseline against synthetic unknowns using identical evaluation sets:
- In-distribution knowns: `X_val_eval` (110 evaluation rows from validation partition).
- Box-uniform unknowns: `box_eval` (1,000 samples, seed 202).
- Column-shuffled unknowns: `shuf_eval` (1,000 samples, seed 404).
- Detection metrics: AUROC and FPR@95% TPR via `evaluate_detector`.


In [11]:
# 1. 1D Range Check Scorer
train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)
train_spread = np.where((train_max - train_min) > 0, train_max - train_min, 1.0)

def compute_range_check_scores(X_raw_input, t_min=train_min, t_max=train_max, t_spread=train_spread):
    low_viol = np.maximum(0.0, (t_min - X_raw_input) / t_spread)
    high_viol = np.maximum(0.0, (X_raw_input - t_max) / t_spread)
    return np.max(np.maximum(low_viol, high_viol), axis=1)

# V0 Scores
m_known_v0 = compute_mahalanobis_scores(X_val_eval_scaled, crop_stats_v0, classes)
m_box_v0 = compute_mahalanobis_scores(scaler_v0.transform(box_eval), crop_stats_v0, classes)
m_shuf_v0 = compute_mahalanobis_scores(scaler_v0.transform(shuf_eval), crop_stats_v0, classes)

auc_box_v0, fpr_box_v0 = evaluate_detector(m_known_v0, m_box_v0)
auc_shuf_v0, fpr_shuf_v0 = evaluate_detector(m_known_v0, m_shuf_v0)

# V2 Scores
m_known_v2 = compute_v2_scores(X_val_eval, scaler_v2_full, crop_stats_v2_full, classes)
m_box_v2 = compute_v2_scores(box_eval, scaler_v2_full, crop_stats_v2_full, classes)
m_shuf_v2 = compute_v2_scores(shuf_eval, scaler_v2_full, crop_stats_v2_full, classes)

auc_box_v2, fpr_box_v2 = evaluate_detector(m_known_v2, m_box_v2)
auc_shuf_v2, fpr_shuf_v2 = evaluate_detector(m_known_v2, m_shuf_v2)

# Range Check Scores
rc_known = compute_range_check_scores(X_val_eval)
rc_box = compute_range_check_scores(box_eval)
rc_shuf = compute_range_check_scores(shuf_eval)

auc_box_rc, fpr_box_rc = evaluate_detector(rc_known, rc_box)
auc_shuf_rc, fpr_shuf_rc = evaluate_detector(rc_known, rc_shuf)

e5_comparison_df = pd.DataFrame([
    {"Detector": "V0 Per-Crop Mahalanobis (Ledoit-Wolf)", "Box AUROC": round(auc_box_v0, 4), "Box FPR@95%": round(fpr_box_v0, 4), "Shuffled AUROC": round(auc_shuf_v0, 4), "Shuffled FPR@95%": round(fpr_shuf_v0, 4)},
    {"Detector": "V2 Noise-Aware Mahalanobis", "Box AUROC": round(auc_box_v2, 4), "Box FPR@95%": round(fpr_box_v2, 4), "Shuffled AUROC": round(auc_shuf_v2, 4), "Shuffled FPR@95%": round(fpr_shuf_v2, 4)},
    {"Detector": "1D Range Check (Naive Baseline)", "Box AUROC": round(auc_box_rc, 4), "Box FPR@95%": round(fpr_box_rc, 4), "Shuffled AUROC": round(auc_shuf_rc, 4), "Shuffled FPR@95%": round(fpr_shuf_rc, 4)},
])

print("=== EXPERIMENT E5 SYNTHETIC UNKNOWN BENCHMARK: BEFORE / AFTER ===")
print(e5_comparison_df.to_string(index=False))


=== EXPERIMENT E5 SYNTHETIC UNKNOWN BENCHMARK: BEFORE / AFTER ===
                             Detector  Box AUROC  Box FPR@95%  Shuffled AUROC  Shuffled FPR@95%
V0 Per-Crop Mahalanobis (Ledoit-Wolf)     0.9996          0.0          0.9778            0.0727
           V2 Noise-Aware Mahalanobis     0.9997          0.0          0.9833            0.0364
      1D Range Check (Naive Baseline)     0.5000          1.0          0.5000            1.0000


## Section 6: Leave-One-Crop-Out (LOCO) Robustness (Before / After)

For each of the 22 target crops:
1. Excludes the held-out crop from training data: 21 remaining crops ($1,470$ training rows).
2. Fits V0 baseline: `StandardScaler` + `LedoitWolf(assume_centered=False)`.
3. Fits V2 noise-aware detector: `make_v2_stats(X_tr_loco, y_tr_loco, crops_loco, profile_cfg)`.
4. Fits 1D range check: `loco_min`, `loco_max`, `loco_spread` on the 21 training crops.
5. In-distribution knowns: validation rows of the 21 known crops (315 rows, $y=0$).
6. Out-of-distribution unknowns: validation rows of the held-out crop (15 rows, $y=1$).
7. Evaluates AUROC and FPR@95% for all three detectors.


In [12]:
loco_comparison_rows = []
all_loco_s_unknown_v0 = []
all_loco_s_unknown_v2 = []

for held_out in classes:
    # 21-crop training partition
    train_mask = (y_train != held_out)
    X_tr_loco = X_train[train_mask]
    y_tr_loco = y_train[train_mask]
    crops_loco = [c for c in classes if c != held_out]

    # 1. Fit V0 LOCO
    sc_loco_v0 = StandardScaler().fit(X_tr_loco)
    X_tr_loco_sc = sc_loco_v0.transform(X_tr_loco)
    stats_loco_v0 = {}
    for c in crops_loco:
        lw = LedoitWolf(assume_centered=False).fit(X_tr_loco_sc[y_tr_loco == c])
        stats_loco_v0[c] = {'mean': lw.location_, 'inv_cov': lw.precision_}

    def score_loco_v0(X_raw):
        X_sc = sc_loco_v0.transform(X_raw)
        min_d = np.full(len(X_sc), np.inf)
        for c in crops_loco:
            diff = X_sc - stats_loco_v0[c]['mean']
            d2 = np.sum((diff @ stats_loco_v0[c]['inv_cov']) * diff, axis=1)
            min_d = np.minimum(min_d, d2)
        return min_d

    # 2. Fit V2 LOCO
    sc_loco_v2, stats_loco_v2 = make_v2_stats(X_tr_loco, y_tr_loco, crops_loco, profile_cfg=PROFILE_CFG)
    def score_loco_v2(X_raw):
        return compute_v2_scores(X_raw, sc_loco_v2, stats_loco_v2, crops_loco)

    # 3. Fit Range Check LOCO
    loco_min = X_tr_loco.min(axis=0)
    loco_max = X_tr_loco.max(axis=0)
    loco_spread = np.where((loco_max - loco_min) > 0, loco_max - loco_min, 1.0)
    def score_loco_rc(X_raw):
        low_viol = np.maximum(0.0, (loco_min - X_raw) / loco_spread)
        high_viol = np.maximum(0.0, (X_raw - loco_max) / loco_spread)
        return np.max(np.maximum(low_viol, high_viol), axis=1)

    val_known_mask = (y_val != held_out)
    val_held_out_mask = (y_val == held_out)
    X_val_known = X_val[val_known_mask]
    X_val_unknown = X_val[val_held_out_mask]

    # Evaluate V0
    s_known_v0 = score_loco_v0(X_val_known)
    s_unknown_v0 = score_loco_v0(X_val_unknown)
    auc_v0, fpr_v0 = evaluate_detector(s_known_v0, s_unknown_v0)
    all_loco_s_unknown_v0.extend(s_unknown_v0)

    # Evaluate V2
    s_known_v2 = score_loco_v2(X_val_known)
    s_unknown_v2 = score_loco_v2(X_val_unknown)
    auc_v2, fpr_v2 = evaluate_detector(s_known_v2, s_unknown_v2)
    all_loco_s_unknown_v2.extend(s_unknown_v2)

    # Evaluate Range Check
    s_known_rc = score_loco_rc(X_val_known)
    s_unknown_rc = score_loco_rc(X_val_unknown)
    auc_rc, fpr_rc = evaluate_detector(s_known_rc, s_unknown_rc)

    diff = auc_v2 - auc_v0
    loco_comparison_rows.append({
        'crop': held_out,
        'V0 AUROC': round(auc_v0, 4),
        'V2 AUROC': round(auc_v2, 4),
        'diff (V2-V0)': round(diff, 4),
        'V0 FPR@95': round(fpr_v0, 4),
        'V2 FPR@95': round(fpr_v2, 4),
        'RC AUROC': round(auc_rc, 4),
        'RC FPR@95': round(fpr_rc, 4)
    })

df_loco_comp = pd.DataFrame(loco_comparison_rows)
print("=== LOCO PER-CROP COMPARISON TABLE (22 CROPS) ===")
print(df_loco_comp.to_string(index=False))

print()
print("=== LOCO SUMMARY STATISTICS ===")
print(f"V0 AUROC:    {df_loco_comp['V0 AUROC'].mean():.4f} +/- {df_loco_comp['V0 AUROC'].std():.4f}")
print(f"V2 AUROC:    {df_loco_comp['V2 AUROC'].mean():.4f} +/- {df_loco_comp['V2 AUROC'].std():.4f}")
print(f"RC AUROC:    {df_loco_comp['RC AUROC'].mean():.4f} +/- {df_loco_comp['RC AUROC'].std():.4f}")
print(f"V0 FPR@95:   {df_loco_comp['V0 FPR@95'].mean():.4f} +/- {df_loco_comp['V0 FPR@95'].std():.4f}")
print(f"V2 FPR@95:   {df_loco_comp['V2 FPR@95'].mean():.4f} +/- {df_loco_comp['V2 FPR@95'].std():.4f}")
print(f"RC FPR@95:   {df_loco_comp['RC FPR@95'].mean():.4f} +/- {df_loco_comp['RC FPR@95'].std():.4f}")

lower_crops_df = df_loco_comp[df_loco_comp['diff (V2-V0)'] < 0]
print()
print(f"=== CROPS WHERE V2 AUROC IS LOWER THAN V0 ({len(lower_crops_df)} CROPS) ===")
print(lower_crops_df[['crop', 'V0 AUROC', 'V2 AUROC', 'diff (V2-V0)', 'V0 FPR@95', 'V2 FPR@95']].to_string(index=False))


=== LOCO PER-CROP COMPARISON TABLE (22 CROPS) ===
       crop  V0 AUROC  V2 AUROC  diff (V2-V0)  V0 FPR@95  V2 FPR@95  RC AUROC  RC FPR@95
      apple    1.0000    1.0000        0.0000     0.0000     0.0000    0.5000        1.0
     banana    1.0000    1.0000        0.0000     0.0000     0.0000    0.5000        1.0
  blackgram    0.9837    0.9964        0.0127     0.0825     0.0254    0.5000        1.0
   chickpea    1.0000    1.0000        0.0000     0.0000     0.0000    0.8667        1.0
    coconut    1.0000    0.9998       -0.0002     0.0000     0.0032    0.7333        1.0
     coffee    1.0000    1.0000        0.0000     0.0000     0.0000    0.5000        1.0
     cotton    0.9987    0.9994        0.0006     0.0159     0.0095    0.6667        1.0
     grapes    1.0000    1.0000        0.0000     0.0000     0.0000    0.5333        1.0
       jute    0.7934    0.7437       -0.0497     0.7587     0.6635    0.5000        1.0
kidneybeans    0.9596    0.9932        0.0337     0.2508    

## Section 7: Box-Uniform Detail

Detailed inspection of detection performance on hyper-rectangle uniform unknowns (`box_eval`, 1,000 samples).


In [13]:
box_detail_df = pd.DataFrame([
    {"Detector": "V0 Per-Crop Mahalanobis (Ledoit-Wolf)", "Box AUROC": round(auc_box_v0, 4), "Box FPR@95%": round(fpr_box_v0, 4)},
    {"Detector": "V2 Noise-Aware Mahalanobis", "Box AUROC": round(auc_box_v2, 4), "Box FPR@95%": round(fpr_box_v2, 4)},
    {"Detector": "1D Range Check (Naive Baseline)", "Box AUROC": round(auc_box_rc, 4), "Box FPR@95%": round(fpr_box_rc, 4)},
])
print("=== BOX-UNIFORM DETECTION PERFORMANCE DETAIL ===")
print(box_detail_df.to_string(index=False))


=== BOX-UNIFORM DETECTION PERFORMANCE DETAIL ===
                             Detector  Box AUROC  Box FPR@95%
V0 Per-Crop Mahalanobis (Ledoit-Wolf)     0.9996          0.0
           V2 Noise-Aware Mahalanobis     0.9997          0.0
      1D Range Check (Naive Baseline)     0.5000          1.0


## Section 8: Per-Profile Operating Rates (Conservative, Balanced, Bold)

We evaluate operating rates under the locked threshold definitions:
- **V0 Thresholds**: refuse cutoffs [11.71 (p97), 12.92 (p99), 13.75 (p99.5)], caution cutoffs [9.53 (p90), 10.78 (p95), 11.71 (p97)].
- **V2 Thresholds**: derived strictly from Section 3 training OOF percentiles:
  - Conservative: caution = 6.19 (p90), refuse = 7.59 (p97)
  - Balanced: caution = 7.06 (p95), refuse = 8.64 (p99)
  - Bold: caution = 7.59 (p97), refuse = 9.18 (p99.5)

**Operating rate definitions:**
- **(a) Clean false-refusal rate**: share of familiar evaluation validation rows (`X_val_eval`, $n=110$) with $\min d^2 > \text{refuse}$.
- **(b) Noisy refusal rate**: share of familiar evaluation validation rows perturbed with `apply_noise(primary_1sigma)` across 5 seeds (1001 to 1005, pooled $n=550$) with $\min d^2 > \text{refuse}$.
- **(c) Unknown pass rate (GATE-ONLY)**: share of box ($n=1,000$) and column-shuffled ($n=1,000$) unknowns NOT refused ($\min d^2 \le \text{refuse}$), and share below caution ($\min d^2 \le \text{caution}$).
- **(d) LOCO held-out pass rate (GATE-ONLY)**: share of pooled held-out validation rows across 22 crops ($n=330$) NOT refused ($\min d^2 \le \text{refuse}$), and share below caution ($\min d^2 \le \text{caution}$).

*(Note: Unknown pass rates are strictly gate-only admission rates, not end-to-end recommendation rates).*


In [14]:
from backend.noise import apply_noise

# 1. Clean distances on X_val_eval (n=110)
d2_clean_v0 = m_known_v0
d2_clean_v2 = m_known_v2

# 2. Noisy distances on X_val_eval across seeds 1001-1005 (n=550)
noisy_eval_batches = []
for s in [1001, 1002, 1003, 1004, 1005]:
    noisy_batch = apply_noise(X_val_eval, n_samples=1, seed=s, profile="primary_1sigma")
    noisy_eval_batches.append(noisy_batch)
X_val_eval_noisy_550 = np.concatenate(noisy_eval_batches, axis=0)

d2_noisy_v0 = compute_mahalanobis_scores(scaler_v0.transform(X_val_eval_noisy_550), crop_stats_v0, classes)
d2_noisy_v2 = compute_v2_scores(X_val_eval_noisy_550, scaler_v2_full, crop_stats_v2_full, classes)

# 3. Unknown distances (n=1,000 each)
d2_box_v0 = m_box_v0
d2_box_v2 = m_box_v2
d2_shuf_v0 = m_shuf_v0
d2_shuf_v2 = m_shuf_v2

# 4. LOCO pooled held-out unknown distances (n=330)
loco_d2_v0_arr = np.array(all_loco_s_unknown_v0)
loco_d2_v2_arr = np.array(all_loco_s_unknown_v2)

# Profiles mapping
profiles_dict = {
    'Conservative': {
        'V0': {'caution': 9.53, 'refuse': 11.71},
        'V2': {'caution': round(v2_p90, 2), 'refuse': round(v2_p97, 2)}
    },
    'Balanced': {
        'V0': {'caution': 10.78, 'refuse': 12.92},
        'V2': {'caution': round(v2_p95, 2), 'refuse': round(v2_p99, 2)}
    },
    'Bold': {
        'V0': {'caution': 11.71, 'refuse': 13.75},
        'V2': {'caution': round(v2_p97, 2), 'refuse': round(v2_p99_5, 2)}
    }
}

profile_rate_records = []

for p_name, p_data in profiles_dict.items():
    for model_name, d_clean, d_noisy, d_box, d_shuf, d_loco in [
        ('V0', d2_clean_v0, d2_noisy_v0, d2_box_v0, d2_shuf_v0, loco_d2_v0_arr),
        ('V2', d2_clean_v2, d2_noisy_v2, d2_box_v2, d2_shuf_v2, loco_d2_v2_arr),
    ]:
        c_thresh = p_data[model_name]['caution']
        r_thresh = p_data[model_name]['refuse']
        
        clean_false_refusal = float(np.mean(d_clean > r_thresh) * 100.0)
        noisy_refusal = float(np.mean(d_noisy > r_thresh) * 100.0)
        
        box_gate_pass = float(np.mean(d_box <= r_thresh) * 100.0)
        box_below_caut = float(np.mean(d_box <= c_thresh) * 100.0)
        
        shuf_gate_pass = float(np.mean(d_shuf <= r_thresh) * 100.0)
        shuf_below_caut = float(np.mean(d_shuf <= c_thresh) * 100.0)
        
        loco_gate_pass = float(np.mean(d_loco <= r_thresh) * 100.0)
        loco_below_caut = float(np.mean(d_loco <= c_thresh) * 100.0)
        
        profile_rate_records.append({
            'Profile': p_name,
            'Model': model_name,
            'Caution / Refuse': f"{c_thresh:.2f} / {r_thresh:.2f}",
            'Clean False Refusal (%)': round(clean_false_refusal, 2),
            'Noisy Refusal (%)': round(noisy_refusal, 2),
            'Box Gate Pass (%) [GATE-ONLY]': round(box_gate_pass, 2),
            'Box Below Caut (%) [GATE-ONLY]': round(box_below_caut, 2),
            'Shuf Gate Pass (%) [GATE-ONLY]': round(shuf_gate_pass, 2),
            'Shuf Below Caut (%) [GATE-ONLY]': round(shuf_below_caut, 2),
            'LOCO Gate Pass (%) [GATE-ONLY]': round(loco_gate_pass, 2),
            'LOCO Below Caut (%) [GATE-ONLY]': round(loco_below_caut, 2),
        })

df_rates_summary = pd.DataFrame(profile_rate_records)
print("=== PER-PROFILE OPERATING RATES (CONSERVATIVE, BALANCED, BOLD) ===")
print(df_rates_summary.to_string(index=False))


=== PER-PROFILE OPERATING RATES (CONSERVATIVE, BALANCED, BOLD) ===
     Profile Model Caution / Refuse  Clean False Refusal (%)  Noisy Refusal (%)  Box Gate Pass (%) [GATE-ONLY]  Box Below Caut (%) [GATE-ONLY]  Shuf Gate Pass (%) [GATE-ONLY]  Shuf Below Caut (%) [GATE-ONLY]  LOCO Gate Pass (%) [GATE-ONLY]  LOCO Below Caut (%) [GATE-ONLY]
Conservative    V0     9.53 / 11.71                     2.73              43.27                            0.2                             0.2                             7.3                              4.4                           11.21                             4.85
Conservative    V2      6.19 / 7.59                     1.82              28.91                            0.3                             0.1                             5.8                              4.0                            9.09                             5.45
    Balanced    V0    10.78 / 12.92                     2.73              38.36                            0.3    

## Section 9: Evaluation Sample Sizes (n) and Evidence Strength Audit

Any evaluation evaluation cohort with $n < 30$ samples is flagged as `thin evidence`.


In [15]:
sample_size_records = [
    {"Metric Category": "Clean Validation False Refusal", "Evaluation Cohort": "X_val_eval (unperturbed)", "Sample Size (n)": len(X_val_eval), "Thin Evidence (<30)": len(X_val_eval) < 30},
    {"Metric Category": "Noisy Validation Refusal", "Evaluation Cohort": "X_val_eval perturbed (5 seeds x 110 rows)", "Sample Size (n)": len(X_val_eval_noisy_550), "Thin Evidence (<30)": len(X_val_eval_noisy_550) < 30},
    {"Metric Category": "Box-Uniform Unknown Pass Rate", "Evaluation Cohort": "box_eval (seed 202)", "Sample Size (n)": len(box_eval), "Thin Evidence (<30)": len(box_eval) < 30},
    {"Metric Category": "Shuffled-Columns Unknown Pass Rate", "Evaluation Cohort": "shuf_eval (seed 404)", "Sample Size (n)": len(shuf_eval), "Thin Evidence (<30)": len(shuf_eval) < 30},
    {"Metric Category": "LOCO Pooled Held-Out Unknown Pass Rate", "Evaluation Cohort": "Pooled held-out validation rows (22 crops x 15 rows)", "Sample Size (n)": len(loco_d2_v0_arr), "Thin Evidence (<30)": len(loco_d2_v0_arr) < 30},
    {"Metric Category": "LOCO Per-Crop Unknown Subsets", "Evaluation Cohort": "Individual crop validation unknown subset (per crop)", "Sample Size (n)": 15, "Thin Evidence (<30)": True},
    {"Metric Category": "X_val_eval Per-Crop Subsets", "Evaluation Cohort": "Individual crop evaluation known subset (per crop)", "Sample Size (n)": 5, "Thin Evidence (<30)": True},
]

df_sample_sizes = pd.DataFrame(sample_size_records)
print("=== EVALUATION SAMPLE SIZES & THIN EVIDENCE AUDIT ===")
print(df_sample_sizes.to_string(index=False))


=== EVALUATION SAMPLE SIZES & THIN EVIDENCE AUDIT ===
                       Metric Category                                    Evaluation Cohort  Sample Size (n)  Thin Evidence (<30)
        Clean Validation False Refusal                             X_val_eval (unperturbed)              110                False
              Noisy Validation Refusal            X_val_eval perturbed (5 seeds x 110 rows)              550                False
         Box-Uniform Unknown Pass Rate                                  box_eval (seed 202)             1000                False
    Shuffled-Columns Unknown Pass Rate                                 shuf_eval (seed 404)             1000                False
LOCO Pooled Held-Out Unknown Pass Rate Pooled held-out validation rows (22 crops x 15 rows)              330                False
         LOCO Per-Crop Unknown Subsets Individual crop validation unknown subset (per crop)               15                 True
           X_val_eval Per-Crop Subse

## Section 10: Full-Pipeline Verdicts (Stages 1–5 Integration)

Evaluates full-pipeline verdicts (Refuse / Caution / Recommend) integrating Stage 2 (OOD Gate under V0 vs V2 thresholds), Stage 3 (Calibrated Probabilities), Stage 4 (Stability Assessment under serving noise: n=200, seed=42), and Stage 5 (Verdict Policy across Conservative, Balanced, and Bold profiles).

In [16]:
from backend.trust_engine import TrustEngine
from backend.noise import apply_noise

# 1. Load TrustEngine and thresholds
engine = TrustEngine.load()
thresholds_cfg = engine.thresholds

# 2. Extract Stage 3/4 thresholds and Section 8 OOD gate thresholds
profile_specs = {
    'Conservative': {
        'min_prob': thresholds_cfg['profiles']['conservative']['min_prob'],
        'min_margin': thresholds_cfg['profiles']['conservative']['min_margin'],
        'min_stability': thresholds_cfg['profiles']['conservative']['min_stability'],
        'V0_c': 9.53,
        'V0_r': 11.71,
        'V2_c': round(v2_p90, 2),
        'V2_r': round(v2_p97, 2),
    },
    'Balanced': {
        'min_prob': thresholds_cfg['profiles']['balanced']['min_prob'],
        'min_margin': thresholds_cfg['profiles']['balanced']['min_margin'],
        'min_stability': thresholds_cfg['profiles']['balanced']['min_stability'],
        'V0_c': 10.78,
        'V0_r': 12.92,
        'V2_c': round(v2_p95, 2),
        'V2_r': round(v2_p99, 2),
    },
    'Bold': {
        'min_prob': thresholds_cfg['profiles']['bold']['min_prob'],
        'min_margin': thresholds_cfg['profiles']['bold']['min_margin'],
        'min_stability': thresholds_cfg['profiles']['bold']['min_stability'],
        'V0_c': 11.71,
        'V0_r': 13.75,
        'V2_c': round(v2_p97, 2),
        'V2_r': round(v2_p99_5, 2),
    }
}

# 3. Compute Stage 3 (probabilities) and Stage 4 (stability) reusing TrustEngine logic
def compute_stages_3_and_4(X_raw):
    df_raw = pd.DataFrame(X_raw, columns=CANONICAL_FEATURES)
    probs_all = engine.pipeline.predict_proba(df_raw)
    classes_arr = np.array(engine.classes)
    top_indices = np.argmax(probs_all, axis=1)
    top_crops = classes_arr[top_indices]
    sorted_probs = np.sort(probs_all, axis=1)
    top_probs = sorted_probs[:, -1]
    second_probs = sorted_probs[:, -2]
    margins = top_probs - second_probs
    all_noisy = [apply_noise(r, n_samples=200, seed=42, profile='primary_1sigma') for r in X_raw]
    all_noisy_df = pd.DataFrame(np.concatenate(all_noisy, axis=0), columns=CANONICAL_FEATURES)
    noisy_preds = engine.pipeline.predict(all_noisy_df).reshape(len(X_raw), 200)
    stabilities = np.mean(noisy_preds == top_crops[:, None], axis=1)
    return top_crops, top_probs, margins, stabilities

# Evaluate Stage 3 & 4 metrics once per evaluation set
tc_box, tp_box, mg_box, st_box = compute_stages_3_and_4(box_eval)
tc_shuf, tp_shuf, mg_shuf, st_shuf = compute_stages_3_and_4(shuf_eval)
tc_clean, tp_clean, mg_clean, st_clean = compute_stages_3_and_4(X_val_eval)
tc_noisy, tp_noisy, mg_noisy, st_noisy = compute_stages_3_and_4(X_val_eval_noisy_550)
y_val_eval_noisy_550 = np.tile(y_val_eval, 5)

# 4. Verdict evaluation rule (TrustEngine Stage 5)
def evaluate_verdicts(min_d2, top_probs, margins, stabilities, c_th, r_th, min_p, min_m, min_s):
    n = len(min_d2)
    verdicts = np.empty(n, dtype=object)
    for i in range(n):
        d2 = min_d2[i]
        if d2 > r_th:
            verdicts[i] = 'refuse'
        elif d2 > c_th or top_probs[i] < min_p or margins[i] < min_m or stabilities[i] < min_s:
            verdicts[i] = 'caution'
        else:
            verdicts[i] = 'recommend'
    ref_rate = float(np.mean(verdicts == 'refuse') * 100.0)
    caut_rate = float(np.mean(verdicts == 'caution') * 100.0)
    rec_rate = float(np.mean(verdicts == 'recommend') * 100.0)
    return verdicts, ref_rate, caut_rate, rec_rate

# Table 1: Unknown Sets Full-Pipeline Verdicts
unknown_records = []
for p_name, cfg in profile_specs.items():
    for m_name, d_box, d_shuf, c_th, r_th in [
        ('V0', d2_box_v0, d2_shuf_v0, cfg['V0_c'], cfg['V0_r']),
        ('V2', d2_box_v2, d2_shuf_v2, cfg['V2_c'], cfg['V2_r']),
    ]:
        _, b_ref, b_caut, b_rec = evaluate_verdicts(d_box, tp_box, mg_box, st_box, c_th, r_th, cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
        unknown_records.append({
            'Profile': p_name, 'Model': m_name, 'Dataset': 'box_eval',
            'Refuse (%)': round(b_ref, 2), 'Caution (%)': round(b_caut, 2), 'Recommend (%)': round(b_rec, 2),
            'n': len(d_box), 'Thin Evidence (<30)': len(d_box) < 30
        })
        _, s_ref, s_caut, s_rec = evaluate_verdicts(d_shuf, tp_shuf, mg_shuf, st_shuf, c_th, r_th, cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
        unknown_records.append({
            'Profile': p_name, 'Model': m_name, 'Dataset': 'shuf_eval',
            'Refuse (%)': round(s_ref, 2), 'Caution (%)': round(s_caut, 2), 'Recommend (%)': round(s_rec, 2),
            'n': len(d_shuf), 'Thin Evidence (<30)': len(d_shuf) < 30
        })

df_unknown_full = pd.DataFrame(unknown_records)
print('=== UNKNOWN SETS FULL-PIPELINE VERDICTS ===')
print(df_unknown_full.to_string(index=False))

# Table 2: Known Validation Clean and Noisy Verdicts + Selective Accuracy
known_records = []
for p_name, cfg in profile_specs.items():
    for m_name, d_cl, d_no, c_th, r_th in [
        ('V0', d2_clean_v0, d2_noisy_v0, cfg['V0_c'], cfg['V0_r']),
        ('V2', d2_clean_v2, d2_noisy_v2, cfg['V2_c'], cfg['V2_r']),
    ]:
        v_cl, cl_ref, cl_caut, cl_rec = evaluate_verdicts(d_cl, tp_clean, mg_clean, st_clean, c_th, r_th, cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
        rec_cl_mask = (v_cl == 'recommend')
        n_cl_rec = int(np.sum(rec_cl_mask))
        sel_acc_cl = float(np.mean(tc_clean[rec_cl_mask] == y_val_eval[rec_cl_mask]) * 100.0) if n_cl_rec > 0 else 0.0
        known_records.append({
            'Profile': p_name, 'Model': m_name, 'Condition': 'Clean (X_val_eval)',
            'Refuse (%)': round(cl_ref, 2), 'Caution (%)': round(cl_caut, 2), 'Recommend (%)': round(cl_rec, 2),
            'n_total': len(d_cl), 'n_rec': n_cl_rec,
            'Selective Accuracy (%)': round(sel_acc_cl, 2),
            'Thin Evidence (<30)': n_cl_rec < 30
        })
        v_no, no_ref, no_caut, no_rec = evaluate_verdicts(d_no, tp_noisy, mg_noisy, st_noisy, c_th, r_th, cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
        rec_no_mask = (v_no == 'recommend')
        n_no_rec = int(np.sum(rec_no_mask))
        sel_acc_no = float(np.mean(tc_noisy[rec_no_mask] == y_val_eval_noisy_550[rec_no_mask]) * 100.0) if n_no_rec > 0 else 0.0
        known_records.append({
            'Profile': p_name, 'Model': m_name, 'Condition': 'Noisy (seeds 1001-1005)',
            'Refuse (%)': round(no_ref, 2), 'Caution (%)': round(no_caut, 2), 'Recommend (%)': round(no_rec, 2),
            'n_total': len(d_no), 'n_rec': n_no_rec,
            'Selective Accuracy (%)': round(sel_acc_no, 2),
            'Thin Evidence (<30)': n_no_rec < 30
        })

df_known_full = pd.DataFrame(known_records)
print()
print('=== KNOWN VALIDATION ROWS FULL-PIPELINE VERDICTS & SELECTIVE ACCURACY ===')
print(df_known_full.to_string(index=False))

# Table 3: Diagnostic Rows (Val Row 203 & Impossible Row)
r203_raw = np.array([[float(df.iloc[203][f]) for f in CANONICAL_FEATURES]])
d2_203_v0 = float(compute_mahalanobis_scores(scaler_v0.transform(r203_raw), crop_stats_v0, classes)[0])
d2_203_v2 = float(compute_v2_scores(r203_raw, scaler_v2_full, crop_stats_v2_full, classes)[0])
pct_203_v0 = float(np.mean(oof_v0_scores <= d2_203_v0) * 100.0)
pct_203_v2 = float(np.mean(oof_v2_scores <= d2_203_v2) * 100.0)

train_medians = df.iloc[train_idx][CANONICAL_FEATURES].median().to_dict()
imp_raw = np.array([[112.0, float(train_medians['P']), 185.0, float(train_medians['temperature']),
                     float(train_medians['humidity']), float(train_medians['ph']), float(train_medians['rainfall'])]])
d2_imp_v0 = float(compute_mahalanobis_scores(scaler_v0.transform(imp_raw), crop_stats_v0, classes)[0])
d2_imp_v2 = float(compute_v2_scores(imp_raw, scaler_v2_full, crop_stats_v2_full, classes)[0])
pct_imp_v0 = float(np.mean(oof_v0_scores <= d2_imp_v0) * 100.0)
pct_imp_v2 = float(np.mean(oof_v2_scores <= d2_imp_v2) * 100.0)

_, tp_203, mg_203, st_203 = compute_stages_3_and_4(r203_raw)
_, tp_imp, mg_imp, st_imp = compute_stages_3_and_4(imp_raw)

case_records = []
for p_name, cfg in profile_specs.items():
    v0_203, _, _, _ = evaluate_verdicts([d2_203_v0], tp_203, mg_203, st_203, cfg['V0_c'], cfg['V0_r'], cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
    v2_203, _, _, _ = evaluate_verdicts([d2_203_v2], tp_203, mg_203, st_203, cfg['V2_c'], cfg['V2_r'], cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
    v0_imp, _, _, _ = evaluate_verdicts([d2_imp_v0], tp_imp, mg_imp, st_imp, cfg['V0_c'], cfg['V0_r'], cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
    v2_imp, _, _, _ = evaluate_verdicts([d2_imp_v2], tp_imp, mg_imp, st_imp, cfg['V2_c'], cfg['V2_r'], cfg['min_prob'], cfg['min_margin'], cfg['min_stability'])
    case_records.append({
        'Case': 'Val Row 203', 'Profile': p_name,
        'V0 min d2': round(d2_203_v0, 4), 'V0 Percentile': round(pct_203_v0, 1), 'V0 Verdict': v0_203[0],
        'V2 min d2': round(d2_203_v2, 4), 'V2 Percentile': round(pct_203_v2, 1), 'V2 Verdict': v2_203[0],
        'n': 1, 'Thin Evidence (<30)': True
    })
    case_records.append({
        'Case': 'Impossible Row', 'Profile': p_name,
        'V0 min d2': round(d2_imp_v0, 4), 'V0 Percentile': round(pct_imp_v0, 1), 'V0 Verdict': v0_imp[0],
        'V2 min d2': round(d2_imp_v2, 4), 'V2 Percentile': round(pct_imp_v2, 1), 'V2 Verdict': v2_imp[0],
        'n': 1, 'Thin Evidence (<30)': True
    })

df_cases_full = pd.DataFrame(case_records)
print()
print('=== VAL ROW 203 & IMPOSSIBLE ROW DIAGNOSTICS & VERDICTS ===')
print(df_cases_full.to_string(index=False))

# Table 4: Complete Evaluation Sample Size and Thin Evidence Summary (All Notebook Sets)
eval_summary_records = [
    {'Evaluation Set / Cohort': 'Training OOF Reference (X_train 5-fold)', 'Sample Size (n)': 1540, 'Thin Evidence (<30)': False, 'Notes': 'Reference percentiles baseline'},
    {'Evaluation Set / Cohort': 'Known Validation Clean Evaluation (X_val_eval)', 'Sample Size (n)': len(X_val_eval), 'Thin Evidence (<30)': len(X_val_eval) < 30, 'Notes': '5 rows per crop x 22 crops'},
    {'Evaluation Set / Cohort': 'Known Validation Noisy Evaluation (seeds 1001-1005)', 'Sample Size (n)': len(X_val_eval_noisy_550), 'Thin Evidence (<30)': len(X_val_eval_noisy_550) < 30, 'Notes': '5 perturbation seeds x 110 clean rows'},
    {'Evaluation Set / Cohort': 'Box-Uniform Unknowns (box_eval, seed 202)', 'Sample Size (n)': len(box_eval), 'Thin Evidence (<30)': len(box_eval) < 30, 'Notes': 'Uniformly sampled across train bounds'},
    {'Evaluation Set / Cohort': 'Shuffled-Columns Unknowns (shuf_eval, seed 404)', 'Sample Size (n)': len(shuf_eval), 'Thin Evidence (<30)': len(shuf_eval) < 30, 'Notes': 'Permuted columns breaking joint correlations'},
    {'Evaluation Set / Cohort': 'LOCO Pooled Held-Out Unknowns', 'Sample Size (n)': len(loco_d2_v0_arr), 'Thin Evidence (<30)': len(loco_d2_v0_arr) < 30, 'Notes': 'Pooled across all 22 held-out folds (15 val rows/crop)'},
    {'Evaluation Set / Cohort': 'LOCO Individual Held-Out Unknowns (per crop)', 'Sample Size (n)': 15, 'Thin Evidence (<30)': True, 'Notes': 'Per-crop validation unknown slice (n=15 < 30)'},
    {'Evaluation Set / Cohort': 'X_val_eval Per-Crop Known Subsets', 'Sample Size (n)': 5, 'Thin Evidence (<30)': True, 'Notes': 'Per-crop evaluation known slice (n=5 < 30)'},
    {'Evaluation Set / Cohort': 'Validation Row 203 Diagnostic Case', 'Sample Size (n)': 1, 'Thin Evidence (<30)': True, 'Notes': 'Single row case study'},
    {'Evaluation Set / Cohort': 'Impossible Row Diagnostic Case', 'Sample Size (n)': 1, 'Thin Evidence (<30)': True, 'Notes': 'Single row case study'},
]

df_eval_summary = pd.DataFrame(eval_summary_records)
print()
print('=== COMPLETE EVALUATION SAMPLE SIZE & THIN EVIDENCE AUDIT (ALL NOTEBOOK SETS) ===')
print(df_eval_summary.to_string(index=False))

C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeClassifier from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle e

C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CalibratedClassifierCV from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


=== UNKNOWN SETS FULL-PIPELINE VERDICTS ===
     Profile Model   Dataset  Refuse (%)  Caution (%)  Recommend (%)    n  Thin Evidence (<30)
Conservative    V0  box_eval        99.8          0.1            0.1 1000                False
Conservative    V0 shuf_eval        92.7          4.4            2.9 1000                False
Conservative    V2  box_eval        99.7          0.2            0.1 1000                False
Conservative    V2 shuf_eval        94.2          3.0            2.8 1000                False
    Balanced    V0  box_eval        99.7          0.2            0.1 1000                False
    Balanced    V0 shuf_eval        90.1          6.0            3.9 1000                False
    Balanced    V2  box_eval        99.5          0.3            0.2 1000                False
    Balanced    V2 shuf_eval        92.1          4.0            3.9 1000                False
        Bold    V0  box_eval        99.5          0.3            0.2 1000                False
      